# Chien luoc (Vu Truong)

- Các chỉ báo sử dụng:
  + EMA34
  + EMA89
- Khung thời gian: H4
- Điều kiện vào lệnh:
Lệnh buy:
  + Đường EMA34 > EMA89
  + Giá đóng cửa > Giá mở cửa (Nến xanh)
  + Giá đóng cửa cắt EMA34 và Giá đóng cửa > EMA34
  + Vào lệnh stop tại cây nến tiếp theo
  + Entry: Lệnh Buystop tại giá High + 10 pip
  + Stoploss: Tại giá Low - 10 pip
  + Take Profit: 2 * (giá High - giá Low)
Lệnh sell:
  + Đường EMA34 < EMA89
  + Giá đóng cửa < Giá mở cửa (Nến đỏ)
  + Giá đóng cửa cắt EMA34 và Giá đóng cửa < EMA34
  + Vào lệnh stop tại cây nến tiếp theo
  + Entry: Lệnh Sellstop tại giá Low - 10 pip
  + Stoploss: Tại giá High + 10 pip
  + Take Profit: 2 * (giá High - giá Low)

In [1]:
# Đường EMA34 > EMA89
#   + Giá đóng cửa > Giá mở cửa (Nến xanh)
#   + Giá đóng cửa cắt EMA34 và Giá đóng cửa > EMA34

# Đường EMA34 < EMA89
#   + Giá đóng cửa < Giá mở cửa (Nến đỏ)
#   + Giá đóng cửa cắt EMA34 và Giá đóng cửa < EMA34

In [2]:
import sys
sys.path.append('../Common/') # Append the path of the Common folder

import CommonBinance # Common functions for Binance

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # Model hồi quy tuyến tính
from sklearn.metrics import mean_squared_error # Đánh giá mô hình tot hay xấu
import talib

# def calculate_momentum(data, period=14):
# 	# data['Momentum'] = data['Close'].diff(period)
# 	# Hoặc Momentum = Giá thị trường hiện tại – Giá trị trung bình trong một khoảng thời gian nhất định
# 	data['Momentum'] = talib.MOM(data['Close'], timeperiod=period)
# 	# Hoặc Momentum = Giá trị hiện tại – Giá trị trước đó

# def calculate_rsi(data, period=14):
# 	# delta = data['Close'].diff(1)
# 	# gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
# 	# loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
# 	# RS = gain / loss
# 	# data['RSI'] = 100 - (100 / (1 + RS))
# 	data['RSI'] = talib.RSI(data['Close'], timeperiod=period)

def detectSignal(symbol, from_date, to_date, timeframe):

	import pandas as pd
	import plotly.graph_objects as go
	import redis
	import numpy as np
	from datetime import datetime

	# ##############################################Step 1: Lấy dữ liệu##############################################
	data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
	# Tra ve Datetime, Open, High, Low, Close, Volume

	# ##############################################Step 2: Chiến lược##############################################  
	# Tính toán các chỉ báo kỹ thuật
	# calculate_momentum(data)
	# calculate_rsi(data)   

	# Đường EMA34 > EMA89
	#   + Giá đóng cửa > Giá mở cửa (Nến xanh)
	#   + Giá đóng cửa cắt EMA34 và Giá đóng cửa > EMA34

	# Đường EMA34 < EMA89
	#   + Giá đóng cửa < Giá mở cửa (Nến đỏ)
	#   + Giá đóng cửa cắt EMA34 và Giá đóng cửa < EMA34

	data['EMA34'] = talib.EMA(data['Close'], timeperiod=34)
	data['EMA89'] = talib.EMA(data['Close'], timeperiod=89)

	# 0 NA -> NA -> NA -> NA -> 10
	# 1 NA -> 10
	# 2 NA -> 10
	# 3 10

	# Đảm bảo không có giá trị NaN trong Momentum, RSI
	# Cach 1: Drop NA
	# data = data.dropna(subset=['Momentum', 'RSI'])

	# Cach 2: Bfill va Ffill
	# Bfill: Backward fill - Điền giá trị NaN bằng giá trị truoc do
	# Ffill: Forward fill - Điền giá trị NaN bằng giá trị sau do
	data['EMA34'] = data['EMA34'].bfill().ffill()
	data['EMA89'] = data['EMA89'].bfill().ffill()

	# => Khong con NaN trong Momentum, RSI
	# ##############################################Step 2: Huấn luyện mô hình hồi quy tuyến tính##############################################
	# Sử dụng Linear Regression để dự đoán giá đóng cửa dựa trên các chỉ báo kỹ thuật Momentum và RSI.
	# X la bien doc lap, y la bien phu thuoc
	# Bien doc lap la X la 1 bien AREA (vi du truoc)
	# Bien phu thuoc la Y la bien Price (vi du truoc)

	# Tạo features (X) và target (y)
	X = data[['Close', 'EMA34', 'EMA89']].shift(1)  # Dùng dữ liệu của ngày hôm trước để dự đoán
	y = data['Close']  # Giá đóng cửa mà chúng ta muốn dự đoán
  
	# Tiếp tục với việc chia dữ liệu
	# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
	# Sử dụng train_test_split để chia dữ liệu thành tập huấn luyện và tập kiểm tra
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Đảm bảo không có giá trị NaN trong X_train và y_train
	# Bfill và Ffill để điền giá trị NaN trong X_train và y_train
	# Nếu có giá trị NaN trong X_train và y_train, mô hình sẽ không thể huấn luyện được
	# Bfill: Backward fill - Điền giá trị NaN bằng giá trị truoc đó
	# Ffill: Forward fill - Điền giá trị NaN bằng giá trị sau đó
	X_train = X_train.bfill().ffill()
	y_train = y_train.bfill().ffill()

	# Huấn luyện mô hình
	model = LinearRegression()
	# Huấn luyện mô hình hồi quy tuyến tính
	model.fit(X_train, y_train)

	# Đảm bảo không có giá trị NaN trong X trước khi dự đoán
	X = X.bfill().ffill()

	# Dự đoán giá đóng cửa sử dụng X đã điền giá trị NaN
	data['Predicted_Close'] = model.predict(X)

	# Có thể sử dụng MSE để đánh giá mô hình ổn thì làm bước kế tiếp
	# Tính Mean Squared Error (MSE) giữa giá trị thực tế và giá trị dự đoán trên tập kiểm tra
	mse = mean_squared_error(data['Close'], data['Predicted_Close'])
	print(f'Mean Squared Error (MSE): {mse}')


	# Đường EMA34 > EMA89
	#   + Giá đóng cửa > Giá mở cửa (Nến xanh)
	#   + Giá đóng cửa cắt EMA34 và Giá đóng cửa > EMA34

	data['Buy_Signal'] = ((data['EMA34'] > data['EMA89']) & data['Close'] > data['Open']) & (data['Close'] >= data['EMA34'])
	data['Sell_Signal'] = ((data['EMA34'] < data['EMA89']) & data['Close'] < data['Open']) & (data['Close'] <= data['EMA34'])

	data.to_csv("OG_FX_ML, Momentum, RSI 2025.06.27.csv")
	
	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line     Buy_Signal      Sell_Signal
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=0)
	# Tạo hash key
	hash_key = 'OG_FX_ML, Momentum, RSI 2025.06.27'
	last_record = data.iloc[-1]
   
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		print(last_record)   
	else:
		print(last_record)
		print('Không có tín hiệu!')

	# ##############################################Step 4: Vẽ biểu đồ##############################################  
	
	# import plotly.graph_objects as go

	# # Đảm bảo rằng data['Datetime'] là dạng datetime nếu chưa phải
	# # data['Datetime'] = pd.to_datetime(data['Datetime'])

	# # Tạo biểu đồ Candlestick cho dữ liệu giá
	# fig = go.Figure(data=[go.Candlestick(x=data['Datetime'],  # Sử dụng cột Datetime thay vì index
	#                                     open=data['Open'],
	#                                     high=data['High'],
	#                                     low=data['Low'],
	#                                     close=data['Close'],
	#                                     name='Candlestick')])

	# # Thêm dòng dự đoán giá đóng cửa từ mô hình hồi quy
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

	# # Thêm điểm mua
	# fig.add_trace(go.Scatter(x=data[data['Buy_Signal']]['Datetime'],
	#                         y=data[data['Buy_Signal']]['Close'],
	#                         mode='markers',
	#                         marker=dict(color='Green', size=10, symbol='triangle-up'),
	#                         name='Buy Signal'))

	# # Thêm điểm bán
	# fig.add_trace(go.Scatter(x=data[data['Sell_Signal']]['Datetime'],
	#                         y=data[data['Sell_Signal']]['Close'],
	#                         mode='markers',
	#                         marker=dict(color='Red', size=10, symbol='triangle-down'),
	#                         name='Sell Signal'))

	# # Thêm Momentum và RSI vào trục Y phụ
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['Momentum'], name='Momentum', yaxis='y2'))
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['RSI'], name='RSI', yaxis='y3'))

	# # Tùy chỉnh layout để thêm trục Y phụ và cập nhật tiêu đề trục X
	# fig.update_layout(
	#     title=f'Trading Signals for {symbol} based on Linear Regression, Momentum, and RSI',
	#     xaxis_title='Date',
	#     yaxis_title='Price',
	#     xaxis_rangeslider_visible=False,
	#     yaxis=dict(
	#         title='Price',
	#         titlefont=dict(color='#1f77b4'),
	#     ),
	#     yaxis2=dict(
	#         title='Momentum',
	#         titlefont=dict(color='#ff7f0e'),
	#         anchor='free',
	#         overlaying='y',
	#         side='left',
	#         position=0.15
	#     ),
	#     yaxis3=dict(
	#         title='RSI',
	#         titlefont=dict(color='#d62728'),
	#         anchor='x',
	#         overlaying='y',
	#         side='right'
	#     ),
	# )

	# fig.show()


In [ ]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_detectSignal(timeframe):
	# Ghi log thoi gian bat dau chay ra man hinh
	print(f"Running detectSignal for timeframe {timeframe} at {datetime.now()}")
	symbol = 'ETHUSDT' # Gia su chung ta danh ETHUSDT trong Timeframe 1m
	from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm nay da du nen
	to_date = (datetime.now() + timedelta(days=0)).strftime('%Y-%m-%d')
	detectSignal(symbol, from_date, to_date, timeframe)

	# Ghi log thoi gian ket thuc chay ra	
	print(f"Finished detectSignal for timeframe {timeframe} at {datetime.now()}")

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_hours = list(range(0,24,1)) # Tu 0 den 59
while True:
	current_time = datetime.now()
	current_hour = current_time.hour
	current_minute = current_time.minute
	current_second = current_time.second
	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_hour in run_hours: # Nếu có, gọi hàm detectSignal
		if current_minute == 43 and current_second == 00: # Chỉ chạy hàm khi giây là 0
			# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
			last_run = getattr(schedule_detectSignal, 'last_run', None) # Lấy phút cuối cùng mà hàm đã chạy
			# Nếu chưa chạy trong phút hiện tại, gọi hàm và lưu lại phút hiện tại
			if last_run is None or last_run != current_minute:
				schedule_detectSignal('5m')
				# Lưu lại phút cuối cùng mà hàm đã chạy
				setattr(schedule_detectSignal, 'last_run', current_minute)

	# Nghỉ 1 giây trước khi kiểm tra lại
	time.sleep(1)

Running detectSignal for timeframe 5m at 2025-06-27 22:43:00.460929
Mean Squared Error (MSE): 13.41693772288802
Datetime           2025-06-27 15:40:00
Open                           2435.48
High                           2441.01
Low                            2434.36
Close                           2437.5
Volume                       2128.7175
EMA34                      2429.754857
EMA89                      2435.332923
Predicted_Close            2435.711345
Buy_Signal                       False
Sell_Signal                      False
Name: 188, dtype: object
Không có tín hiệu!
Finished detectSignal for timeframe 5m at 2025-06-27 22:43:20.902912
